In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
openai_client = OpenAI()


In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return response.output_text

In [5]:
llm("Write")

'Sure — what would you like me to write?'

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [9]:
# Q1: How many lesson pages are in the dataset?

len(files)

doc = files[0].parse()

for field in doc:
    print(field)

content
filename


In [10]:
from minsearch import Index

# Create documents
# docs = [
#     {
#         "question": "How do I join the course after it has started?",
#         "text": "You can join the course at any time. We have recordings available.",
#         "section": "General Information",
#         "course": "data-engineering-zoomcamp"
#     },
#     {
#         "question": "What are the prerequisites for the course?",
#         "text": "You need to have basic knowledge of programming.",
#         "section": "Course Requirements",
#         "course": "data-engineering-zoomcamp"
#     }
# ]

docs = []

for file in files:
    doc = file.parse()
    docs.append(doc)

    
# Create and fit the index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(docs)


In [15]:
# Q2. Indexing and searching

q = "How does the agentic loop keep calling the model until it stops?"

results = index.search(q)

print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


In [26]:
# Q3. RAG

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'content': 3.0, 'filename': 0.5}
        # filter_dict = {'course': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict
            # filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            # lines.append(doc['section']) 
            lines.append('filename: ' + doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        # return response.output_text
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer


In [27]:
rag = RAGBase(index=index, llm_client=openai_client)


In [28]:
response = rag.rag("How does the agentic loop keep calling the model until it stops?")

print(response)
for f in response:
    print(f)

Response(id='resp_0ba28a1c77d5f32a006a39e76971dc81928bd9e0109cf3d683', created_at=1782179689.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0ba28a1c77d5f32a006a39e76b39248192acece0cbc931f509', content=[ResponseOutputText(annotations=[], text='It keeps calling the model inside a `while True` loop.\n\nAfter each model response, the code checks whether there were any `function_call` items. If there were, it runs the tool, appends the tool output to the message history, and loops again. It stops when a response comes back with no function calls:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the model keeps getting called until it returns a final answer without asking for more tools.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto

In [29]:
print(response.usage)

ResponseUsage(input_tokens=7136, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=103, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7239)


In [30]:
# Q4. CHUNKING

from gitsource import chunk_documents

chunks = chunk_documents(docs, size=2000, step=1000)

len(chunks)

295

In [33]:
# Q5. RAG with chunking

# Create and fit the index
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunked_index.fit(chunks)



In [34]:
chunked_rag = RAGBase(index=index, llm_client=openai_client)


In [37]:
chunked_response = chunked_rag.rag("How does the agentic loop keep calling the model until it stops?")

print(chunked_response.usage)

ResponseUsage(input_tokens=7136, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7226)
